**assignment 3 - NLPROC-ALV-B**

**Anja Kieburg, Friedemann Decker**

## Task 2 Programming

### Task 2 TIGER Corpus

Korpus TIGER 2.2.doc (April 2018)
from: https://www.ims.uni-stuttgart.de/en/research/resources/corpora/tiger/

TIGER Corpus
The TIGER Corpus consists of approximate 900,000 tokens (50,000 sentences) of German newspaper text, taken from the Frankfurter Rundschau. The corpus was semi-automatically POS-tagged and annotated with syntactic structure. Moreover, it contains morphological and lemma information for terminal nodes

In [1]:
# Setup AI
%load_ext jupyter_ai_magics
%load_ext dotenv
%dotenv

In [34]:
import pandas as pd
import numpy as np

**Step 1**: Import TIGER XML into Pandas, using only the entries under `s=Sätze`

In [9]:
tiger_corpus = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//s/graph/terminals/t",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [19]:
len_tiger_corpus = len(tiger_corpus)
len(tiger_corpus)

888238

In [11]:
tiger_corpus.head()

,id,word,lemma,pos,morph,case,number,gender,person,degree,tense,mood,secedge
0,s1_1,``,--,$(,--,--,--,--,--,--,--,--,NaN
1,s1_2,Ross,Ross,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
2,s1_3,Perot,Perot,NE,Nom.Sg.Masc,Nom,Sg,Masc,--,--,--,--,NaN
3,s1_4,wäre,sein,VAFIN,3.Sg.Past.Subj,--,Sg,--,3,--,Past,Subj,NaN
4,s1_5,vielleicht,vielleicht,ADV,--,--,--,--,--,--,--,--,NaN


### Task 2 A) tag t that maximizes p(t|w)

#### Step 2: Required fields: Import "word" and "pos" into the DataFrame

In [12]:
words = tiger_corpus[["word", "pos"]]
words.head()

,word,pos
0,``,$(
1,Ross,NE
2,Perot,NE
3,wäre,VAFIN
4,vielleicht,ADV


#### Step 3: Count the number of unique combinations of "word" AND "pos" = frequency of occurrence in the corpus

In [13]:
word_pos_freq = words[["word", "pos"]].value_counts()
words = word_pos_freq.reset_index(name="pos_freq")
words.head()

,word,pos,pos_freq
0,",","$,",43656
1,.,$.,41057
2,der,ART,25286
3,die,ART,21209
4,und,KON,16281


#### Step 4: Convert "word" to lowercase so that different parts of speech (POS) can be grouped into a "word_lower" category

In [14]:
word_lower = words["word"].str.lower()
words.insert(1, "word_lower", word_lower)
words.head()

,word,word_lower,pos,pos_freq
0,",",",","$,",43656
1,.,.,$.,41057
2,der,der,ART,25286
3,die,die,ART,21209
4,und,und,KON,16281


#### Step 5: Determine the number of words per "word_lower" category and enter it as "word_freq" in each row to calculate p(t|w).

In [15]:
word_freq = words.groupby("word_lower")["pos_freq"].transform("sum")
words.insert(4, "word_freq", word_freq)
words.head()

,word,word_lower,pos,pos_freq,word_freq
0,",",",","$,",43656,43659
1,.,.,$.,41057,41057
2,der,der,ART,25286,29662
3,die,die,ART,21209,30024
4,und,und,KON,16281,16786


#### Step 6: Calculate p_pos = p(t|w)

In [16]:
p_pos = words["pos_freq"] / words["word_freq"]
words.insert(5, "p_pos", p_pos)
words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos
0,",",",","$,",43656,43659,0.999931
1,.,.,$.,41057,41057,1.000000
2,der,der,ART,25286,29662,0.852471
3,die,die,ART,21209,30024,0.706402
4,und,und,KON,16281,16786,0.969915


#### Step 7: Keep only p_pos_max values in the DataFrame

In [17]:
words["max_p_pos"] = words.groupby("word_lower")["p_pos"].transform("max")
words = words[words["p_pos"] == words["max_p_pos"]]

words.head()

,word,word_lower,pos,pos_freq,word_freq,p_pos,max_p_pos
0,",",",","$,",43656,43659,0.999931,0.999931
1,.,.,$.,41057,41057,1.000000,1.000000
2,der,der,ART,25286,29662,0.852471,0.852471
3,die,die,ART,21209,30024,0.706402,0.706402
4,und,und,KON,16281,16786,0.969915,0.969915


#### Step 8: Delete columns that aren't needed (clean up). Rename the columns according to the assignment.

In [72]:
words = words.drop(columns = ["word_lower", "p_pos"])


In [78]:
words = words.rename(columns={'word': 'w', 'pos': 't', 'max_p_pos': 'p'})

**Result**: Computation of the tag **t** for each word **w** that maximizes **p(t|w)**.
List of the first 30 values:

In [79]:
words.head(30)

,w,t,pos_freq,word_freq,p
0,",","$,",43656,43659,0.999931
1,.,$.,41057,41057,1.000000
2,der,ART,25286,29662,0.852471
3,die,ART,21209,30024,0.706402
4,und,KON,16281,16786,0.969915
5,in,APPR,13711,15102,0.907893
6,den,ART,9486,9955,0.952888
7,'',$(,8811,8811,1.000000
8,``,$(,8809,8809,1.000000
9,von,APPR,8182,9033,0.905790


#### Appendix: Meaning of the Abbreviations Used for Parts of Speech (POS)

In [80]:
tiger_corpus_pos_abr = pd.read_xml("tiger_release_aug07.corrected.16012013.xml",
                  xpath="//head/annotation/feature/value",
                  elems_only=False,
                  encoding="ISO-8859-1")

In [81]:
tiger_corpus_pos_abr.head(56)

,name,value
0,$(,Sonstige Satzzeichen; satzintern
1,"$,",Komma
2,$.,Satzbeendende Interpunktion
3,--,<Nicht zugeordnet>
4,ADJA,Attributives Adjektiv
5,ADJD,Adverbiales oder prädikatives Adjektiv
6,ADV,Adverb
7,APPO,Postposition
8,APPR,Präposition; Zirkumposition links
9,APPRART,Präposition mit Artikel


### Task 2 B) - HMM tagger

#### Step 1) Split the corpus into a training set and test set.

In [7]:
%%ai gemini:gemini-3.8-flash
kurz, knapp, zitierfähig: Welche Splitting-Ratio empfiehlst du für einen NLP-Datensatz mit 900.000 Einträgen?

**Empfohlene Aufteilung: 98 : 1 : 1**

* **Train (98 %):** 882.000 Texte
* **Validierung (1 %):** 9.000 Texte
* **Test (1 %):** 9.000 Texte

**Zitierfähige Begründung:**
> *„Bei großen Datensätzen (Big Data, $N \ge 10^5$) weicht die moderne Machine-Learning-Praxis von klassischen 70/30-Verhältnissen ab: Ein Anteil von 1 % ($N = 9.000$) liefert eine statistisch hinreichend große Stichprobe mit engen Konfidenzintervallen für Validierung und Test, während 98 % des Datenvolumens für das datenintensive Training von Sprachmodellen maximiert werden (in Anlehnung an Ng, 2017).*“

In [59]:
# SOLL Anzahl Worte
len_training_set = int(len_tiger_corpus * 0.98)
len_test_set = len_tiger_corpus - len_training_set
print(f"Words in training set calc: {len_training_set}")
print(f"Words in test set calc: {len_test_set}")

Words in training set calc: 870473
Words in test set calc: 17765


In [60]:
#IST Anzahl Worte
training_set = tiger_corpus.iloc[:len_training_set]
training_set = training_set[["word", "pos"]]
test_set = tiger_corpus.iloc[len_training_set:]
test_set = test_set[["word", "pos"]]
print(f"Words in training set act: {len(training_set)}")
print(f"Words in test set act: {len(test_set)}")

Words in training set act: 870473
Words in test set act: 17765


#### Step 2) Counting of POS tags, POS tag - POS tag follower, word - POS tag combination

From the training set, count how often each (i) POS tag occurs,
how often one (ii) POS tag follows another and how often each (iii) word occurs with each POS tag.

In [64]:
# Step 2 (i) Occurence of POS tags
training_set_pos_count = training_set[["pos"]].value_counts()
training_set_pos_count

pos    
NN         180222
ART         95267
APPR        73916
ADJA        53358
NE          51020
$.          44577
$,          42839
ADV         38654
VVFIN       34971
$(          29724
VAFIN       24081
KON         21636
ADJD        18150
VVPP        17332
CARD        15635
APPRART     14528
PPER        13321
VVINF       13150
VMFIN        8629
KOUS         6997
PPOSAT       6462
PRELS        6136
PIAT         6112
PRF          5631
PTKNEG       5203
PTKVZ        4990
PROAV        4887
PIS          4764
PTKZU        4355
PDS          3190
VAINF        3105
PDAT         2870
KOKOM        2615
PWAV         1707
VVIZU        1492
TRUNC        1335
VAPP         1324
FM           1251
XY            997
KOUI          933
PWS           824
VMINF         495
PTKA          435
APZR          351
PRELAT        291
APPO          250
PWAT          181
VVIMP         154
PTKANT         80
ITJ            19
PPOSS          14
VMPP            8
VAIMP           3
NNE             2
Name: count, dtype: 

In [61]:
# Step 2 (ii) POS tag follows another
# Testset mit Follower bauen
# Follower Spalte bauen, alle Zeilen um 1 Zeile nach oben, letzte Zeile = "NaN"
training_set_foll = pd.DataFrame()
training_set_foll[["word_foll", "pos_foll"]] = training_set[["word", "pos"]]
training_set_foll = training_set_foll.iloc[1:].reset_index(drop=True)
training_set_foll.loc[len(training_set_foll)] = np.nan

# Follower Spalte zu Testset hinzufügen
training_set = pd.concat([training_set, training_set_foll], axis=1)

In [62]:
training_set

,word,pos,word_foll,pos_foll
0,``,$(,Ross,NE
1,Ross,NE,Perot,NE
2,Perot,NE,wäre,VAFIN
3,wäre,VAFIN,vielleicht,ADV
4,vielleicht,ADV,ein,ART
...,...,...,...,...
870468,Ruhe,NN,aus,PTKVZ
870469,aus,PTKVZ,.,$.
870470,.,$.,Und,KON
870471,Und,KON,es,PPER


In [66]:
training_set_pos_pos_foll_count = training_set[["pos", "pos_foll"]].value_counts()
training_set_pos_pos_foll_count

pos    pos_foll
ART    NN          60253
ADJA   NN          46199
NN     APPR        32188
APPR   ART         28487
ART    ADJA        23308
                   ...  
$(     VAIMP           1
KOKOM  PRELS           1
VVIMP  PTKZU           1
PDAT   PPOSAT          1
ITJ    ITJ             1
Name: count, Length: 1484, dtype: int64

In [68]:
%%ai gemini:gemini-3.8-flash
Mit welcher Pivot-Pandas/Numpy Funktion(en) formt man training_set_pos_pos_fol um, sodass "pos" die Zeilenbezeichnung,
"pos_foll" die Spaltenbezeichnung und die Zellinhalte die Werte aus value_counts enthält?:
training_set_pos_pos_foll_count = training_set[["pos", "pos_foll"]].value_counts()

Die passendste und direkteste Pandas-Funktion hierfür ist **`.unstack()`**:

```python
# Direkt auf dem Series-Objekt aus value_counts():
result = training_set_pos_pos_foll_count.unstack(fill_value=0)
```

---

### Alternativen:

1. **Mit `.pivot()`** (erfordert vorheriges `.reset_index()`):
   ```python
   result = (
       training_set_pos_pos_foll_count
       .reset_index(name="count")
       .pivot(index="pos", columns="pos_foll", values="count")
       .fillna(0)
   )
   ```

2. **Eleganter Direktweg (ohne vorheriges `value_counts()`):**
   ```python
   result = pd.crosstab(training_set["pos"], training_set["pos_foll"])
   ```

In [74]:
training_set_pos_pos_foll_count_cross = pd.crosstab(training_set["pos"], training_set["pos_foll"])
training_set_pos_pos_foll_count_cross

pos_foll,$(,"$,",$.,ADJA,ADJD,ADV,APPO,APPR,APPRART,APZR,...,VAPP,VMFIN,VMINF,VMPP,VVFIN,VVIMP,VVINF,VVIZU,VVPP,XY
pos,,,,,,,,,,,,,,,,,,,,,
$(,696,2299,2701,1840,811,1338,1,2221,416,1,...,47,217,3,0,1080,73,324,57,681,308
"$,",586,3,0,1787,624,2159,0,3237,440,0,...,1,832,0,0,4974,22,75,26,333,20
$.,3394,13,5,860,1078,4317,0,5806,1262,0,...,0,79,0,0,203,41,27,5,290,224
ADJA,871,749,68,2301,25,55,0,131,20,2,...,2,10,0,0,83,0,26,2,39,1
ADJD,680,1521,1713,1532,804,659,0,2195,315,1,...,121,95,0,2,1037,0,687,101,1528,13
ADV,561,947,712,1853,3113,4648,0,5812,1082,17,...,15,513,0,0,2417,3,684,89,1221,5
APPO,11,7,9,3,12,27,0,32,6,0,...,0,12,0,0,36,0,3,1,6,0
APPR,809,12,13,5407,708,1660,0,263,290,0,...,0,0,0,0,2,2,1,0,0,5
APPRART,145,1,1,3091,28,44,0,6,1,0,...,0,0,0,0,0,0,0,0,0,0


In [69]:
# Step 2 (iii) Occurence of word with each POS tag
training_set_pos_count = training_set[["word", "pos"]].value_counts()
training_set_pos_count

word               pos  
,                  $,       42839
.                  $.       40188
der                ART      24773
die                ART      20762
und                KON      15960
                            ...  
weile              VVFIN        1
Dienstreise        NN           1
tollen             ADJA         1
Regierungsviertel  NN           1
schläfrige         ADJA         1
Name: count, Length: 92812, dtype: int64

c) Compute the transition and observation probabilities of the HMM tagger from this.

d) Then implement the Viterbi algorithm so you can decode a test sentence.

e) Create a Viterbi table
that stores, for each position in the sentence and each possible tag, the probability of the most
probable path ending in that tag.

f) Initialize the table for the first word using the initial tag probabilities and the observation probability of that word.

g) Process the remaining words from left to right.

h) For each word, update the Viterbi table by considering all possible tags and the probabilities of the most probable paths leading to them.

i) Store both the best score and the preceding tag that achieved it (the backpointer).

j) After filling the table, backtrack from the last word to find the most probable sequence of tags.

k) Now run your algorithm on the test set. Report its error rate and compare its performance
to the most frequent tag baseline.